# Proyek Analisis Data: Brazilian E-Commerce Public Dataset by Olist

- **Nama:** Nabil Musannif Siregar
- **Email:** nabilmusannifs@gmail.com
- **ID Dicoding:** nabilmuss

Notebook ini menganalisis tren penjualan, performa kategori produk, dan hubungan ketepatan pengiriman dengan kepuasan pelanggan.

## Menentukan Pertanyaan Bisnis

1. **Bagaimana tren total nilai penjualan produk dan jumlah pesanan setiap bulan pada September 2016–Agustus 2018, dan bulan penuh mana yang mengalami penurunan penjualan terbesar dibandingkan bulan sebelumnya?** Hasilnya digunakan untuk menentukan periode yang perlu dievaluasi dan menghindari kesimpulan dari bulan terakhir yang tidak lengkap.
2. **Kategori produk apa yang memiliki total nilai penjualan dan jumlah pesanan tertinggi serta terendah pada September 2016–Agustus 2018?** Hasilnya digunakan untuk menentukan prioritas stok, promosi, dan evaluasi kategori.
3. **Berapa persentase pesanan yang terlambat setiap bulan, dan state pelanggan mana yang memiliki tingkat keterlambatan tertinggi (minimal 30 pesanan) pada September 2016–Agustus 2018?** Hasilnya digunakan untuk menetapkan prioritas perbaikan SLA logistik.

## Import Semua Packages/Library yang Digunakan

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_DIR = Path.cwd() / "data"
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / "data"
assert DATA_DIR.exists(), f"Folder data tidak ditemukan: {DATA_DIR}"

## Data Wrangling


### Gathering Data

Memuat seluruh dataset yang diperlukan untuk analisis.


In [ ]:
file_names = {
    "customers": "customers_dataset.csv",
    "geolocation": "geolocation_dataset.csv",
    "order_items": "order_items_dataset.csv",
    "order_payments": "order_payments_dataset.csv",
    "order_reviews": "order_reviews_dataset.csv",
    "orders": "orders_dataset.csv",
    "product_category": "product_category_name_translation.csv",
    "products": "products_dataset.csv",
    "sellers": "sellers_dataset.csv",
}
raw = {name: pd.read_csv(DATA_DIR / filename) for name, filename in file_names.items()}
customers_df = raw["customers"].copy()
geolocation_df = raw["geolocation"].copy()
order_items_df = raw["order_items"].copy()
order_payments_df = raw["order_payments"].copy()
order_reviews_df = raw["order_reviews"].copy()
orders_df = raw["orders"].copy()
product_category_df = raw["product_category"].copy()
products_df = raw["products"].copy()
sellers_df = raw["sellers"].copy()
print(f"{len(raw)} dataset berhasil dimuat dari {DATA_DIR}")

9 dataset berhasil dimuat dari /Users/musannif/Documents/Dicoding Era/Belajar Fundamental Analisis Data/submission/data


**Insight:** Sembilan file dataset berhasil dimuat dari folder `data/`. Data yang digunakan untuk analisis utama adalah `orders`, `order_items`, `products`, `product_category_name_translation`, `customers`, dan `order_reviews`.

### Assessing Data

Pemeriksaan berikut mencatat minimal dua masalah kualitas data beserta rencana penanganannya.

In [ ]:
datasets = raw
assessment = []
for name, df in datasets.items():
    assessment.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    })
assessment_df = pd.DataFrame(assessment).sort_values("missing_cells", ascending=False)
display(assessment_df)

print("Masalah yang ditangani:")
print("1. Geolocation memiliki baris duplikat; baris duplikat dihapus agar agregasi lokasi tidak menggandakan data.")
print("2. Products memiliki missing value pada kategori dan atribut numerik; kategori diisi 'Unknown', atribut numerik diisi median.")
print("3. Orders memiliki tanggal pengiriman kosong pada pesanan yang belum selesai; baris tersebut dipertahankan dan dikeluarkan hanya dari analisis SLA.")
print("4. Kolom tanggal pada orders, order_items, dan order_reviews masih bertipe object; dikonversi ke datetime.")

,dataset,rows,columns,missing_cells,duplicate_rows
4,order_reviews,99224,7,145903,0
5,orders,99441,8,4908,0
7,products,32951,9,2448,0
0,customers,99441,5,0,0
1,geolocation,1000163,5,0,261831
2,order_items,112650,7,0,0
3,order_payments,103886,5,0,0
6,product_category,71,2,0,0
8,sellers,3095,4,0,0


Masalah yang ditangani:
1. Geolocation memiliki baris duplikat; baris duplikat dihapus agar agregasi lokasi tidak menggandakan data.
2. Products memiliki missing value pada kategori dan atribut numerik; kategori diisi 'Unknown', atribut numerik diisi median.
3. Orders memiliki tanggal pengiriman kosong pada pesanan yang belum selesai; baris tersebut dipertahankan dan dikeluarkan hanya dari analisis SLA.
4. Kolom tanggal pada orders, order_items, dan order_reviews masih bertipe object; dikonversi ke datetime.


### Cleaning Data

In [ ]:
# 1. Hapus duplikat geolocation.
geolocation_df = geolocation_df.drop_duplicates().copy()

# 2. Tangani missing value products sesuai jenis kolom.
products_df = products_df.copy()
products_df["product_category_name"] = products_df["product_category_name"].fillna("Unknown")
product_numeric_cols = [
    "product_name_lenght", "product_description_lenght", "product_photos_qty",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
]
for column in product_numeric_cols:
    products_df[column] = products_df[column].fillna(products_df[column].median())

# 3. Missing komentar review bukan kegagalan transaksi, sehingga diberi label eksplisit.
order_reviews_df = order_reviews_df.copy()
for column in ["review_comment_title", "review_comment_message"]:
    order_reviews_df[column] = order_reviews_df[column].fillna("No comment")

# 4. Konversi seluruh kolom tanggal dan pertahankan NaT untuk tanggal yang memang belum tersedia.
date_columns = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}
for column in date_columns["orders"]:
    orders_df[column] = pd.to_datetime(orders_df[column], errors="coerce")
for column in date_columns["order_items"]:
    order_items_df[column] = pd.to_datetime(order_items_df[column], errors="coerce")
for column in date_columns["order_reviews"]:
    order_reviews_df[column] = pd.to_datetime(order_reviews_df[column], errors="coerce")

print("Duplikat geolocation setelah cleaning:", geolocation_df.duplicated().sum())
print("Missing products setelah cleaning:", int(products_df.isna().sum().sum()))
print("Tipe order_purchase_timestamp:", orders_df["order_purchase_timestamp"].dtype)

Duplikat geolocation setelah cleaning: 0
Missing products setelah cleaning: 0
Tipe order_purchase_timestamp: datetime64[us]


**Insight:** Duplikat geolocation dihapus, atribut products yang numerik diimputasi dengan median, kategori kosong diberi label `Unknown`, dan kolom tanggal dikonversi ke datetime. Missing date pada orders tidak diisi secara sembarang karena `NaT` dapat berarti pesanan belum selesai atau dibatalkan; kolom tersebut difilter saat menghitung SLA.

## Exploratory Data Analysis (EDA)

In [ ]:
PERIOD_START = pd.Timestamp("2016-09-01")
PERIOD_END = pd.Timestamp("2018-08-31 23:59:59")
orders_period = orders_df[orders_df["order_purchase_timestamp"].between(PERIOD_START, PERIOD_END)].copy()
period_order_ids = set(orders_period["order_id"])
print("Periode analisis:", PERIOD_START.date(), "hingga", PERIOD_END.date())
print("Jumlah pesanan:", orders_period["order_id"].nunique())
display(orders_period[["order_id", "order_status", "order_purchase_timestamp"]].head())

Periode analisis: 2016-09-01 hingga 2018-08-31
Jumlah pesanan: 99421


,order_id,order_status,order_purchase_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39


## Visualization & Explanatory Analysis

Visualisasi berikut menjawab setiap pertanyaan bisnis dengan grafik yang sesuai.


### Pertanyaan 1: Tren Penjualan dan Pesanan

In [ ]:
q1_data = orders_period[["order_id", "order_purchase_timestamp"]].merge(
    order_items_df[["order_id", "price"]], on="order_id", how="left"
)
q1_data["month"] = q1_data["order_purchase_timestamp"].dt.to_period("M")
monthly_sales = q1_data.groupby("month", as_index=False).agg(
    total_sales=("price", "sum"), total_orders=("order_id", "nunique")
)
monthly_sales["month"] = monthly_sales["month"].dt.to_timestamp()
monthly_sales["sales_change"] = monthly_sales["total_sales"].diff()
monthly_sales["sales_change_pct"] = monthly_sales["total_sales"].pct_change() * 100
display(monthly_sales)

,month,total_sales,total_orders,sales_change,sales_change_pct
0,2016-09-01,267.36,4,NaN,NaN
1,2016-10-01,49507.66,324,49240.30,1.841723e+04
2,2016-12-01,10.90,1,-49496.76,-9.997798e+01
3,2017-01-01,120312.87,800,120301.97,1.103688e+06
4,2017-02-01,247303.02,1780,126990.15,1.055499e+02
5,2017-03-01,374344.30,2682,127041.28,5.137069e+01
6,2017-04-01,359927.23,2404,-14417.07,-3.851286e+00
7,2017-05-01,506071.14,3700,146143.91,4.060374e+01
8,2017-06-01,433038.60,3245,-73032.54,-1.443128e+01
9,2017-07-01,498031.48,4026,64992.88,1.500857e+01


In [ ]:
# Bandingkan hanya bulan yang tersedia dalam periode utama; bulan terakhir 2018 tidak ikut agar tidak bias karena data tidak lengkap.
largest_decrease = monthly_sales.loc[monthly_sales["sales_change"].idxmin()]
print(f"Penurunan terbesar: {largest_decrease['month']:%B %Y}")
print(f"Nilai penurunan: R$ {abs(largest_decrease['sales_change']):,.2f}")
print(f"Persentase perubahan: {largest_decrease['sales_change_pct']:.2f}%")

Penurunan terbesar: December 2017
Nilai penurunan: R$ 266,357.20
Persentase perubahan: -26.36%


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(monthly_sales["month"], monthly_sales["total_sales"], marker="o", color="#2563eb", label="Total penjualan")
ax1.set_xlabel("Bulan")
ax1.set_ylabel("Total penjualan (R$)")
ax1.set_title("Tren Total Penjualan dan Jumlah Pesanan per Bulan")
ax1.tick_params(axis="x", rotation=45)
ax2 = ax1.twinx()
ax2.plot(monthly_sales["month"], monthly_sales["total_orders"], marker="s", linestyle="--", color="#f97316", label="Jumlah pesanan")
ax2.set_ylabel("Jumlah pesanan")
handles1, labels1 = ax1.get_legend_handles_labels(); handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper left")
fig.tight_layout()
plt.show()

/var/folders/69/x82gzmxd4f5_74mx5vv2423w0000gn/T/ipykernel_12207/1246945785.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Insight Pertanyaan 1:** Penjualan dan pesanan meningkat kuat sepanjang 2017 hingga awal 2018. Penurunan terbesar pada tabel terjadi pada **September 2018**, turun **R$ 854.541,33 (-99,98%)** dari Agustus, tetapi bulan tersebut hanya memiliki 16 pesanan sehingga bukan bulan penuh. Karena itu, penurunan tersebut tidak boleh dianggap sebagai penurunan performa bisnis tanpa data periode lengkap.

### Pertanyaan 2: Performa Kategori Produk

In [ ]:
q2_data = order_items_df[order_items_df["order_id"].isin(period_order_ids)].merge(
    products_df[["product_id", "product_category_name"]], on="product_id", how="left"
).merge(product_category_df, on="product_category_name", how="left")
q2_data["category"] = q2_data["product_category_name_english"].fillna(q2_data["product_category_name"]).fillna("Unknown")
category_sales = q2_data.groupby("category", as_index=False).agg(
    total_sales=("price", "sum"), total_orders=("order_id", "nunique")
)
category_sales = category_sales.sort_values("total_sales", ascending=False)
top_sales = category_sales.head(10)
bottom_sales = category_sales.tail(10).sort_values("total_sales")
display(category_sales.head(10))
display(category_sales.sort_values("total_orders", ascending=False).head(10))

,category,total_sales,total_orders
44,health_beauty,1258681.34,8836
73,watches_gifts,1205005.68,5624
8,bed_bath_table,1036988.68,9417
68,sports_leisure,988048.97,7720
16,computers_accessories,911954.32,6689
40,furniture_decor,729762.49,6449
21,cool_stuff,635290.85,3632
50,housewares,632248.66,5884
6,auto,592720.11,3897
43,garden_tools,485256.46,3518


,category,total_sales,total_orders
8,bed_bath_table,1036988.68,9417
44,health_beauty,1258681.34,8836
68,sports_leisure,988048.97,7720
16,computers_accessories,911954.32,6689
40,furniture_decor,729762.49,6449
50,housewares,632248.66,5884
73,watches_gifts,1205005.68,5624
71,telephony,323667.53,4199
6,auto,592720.11,3897
72,toys,483946.60,3886


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
sns.barplot(data=top_sales, x="total_sales", y="category", ax=axes[0], color="#2563eb")
axes[0].set_title("10 Kategori dengan Penjualan Tertinggi")
axes[0].set_xlabel("Total penjualan (R$)"); axes[0].set_ylabel("Kategori")
sns.barplot(data=category_sales.sort_values("total_orders", ascending=False).head(10), x="total_orders", y="category", ax=axes[1], color="#14b8a6")
axes[1].set_title("10 Kategori dengan Jumlah Pesanan Tertinggi")
axes[1].set_xlabel("Jumlah pesanan"); axes[1].set_ylabel("Kategori")
fig.tight_layout(); plt.show()

/var/folders/69/x82gzmxd4f5_74mx5vv2423w0000gn/T/ipykernel_12207/275378375.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


**Insight Pertanyaan 2:** `health_beauty` memiliki nilai penjualan tertinggi (**R$ 1.258.681,34**), sedangkan `bed_bath_table` memiliki jumlah pesanan tertinggi (**9.417**). `security_and_services` menjadi yang terendah pada kedua metrik, dengan penjualan **R$ 283,29** dan **2 pesanan**. Temuan ini mendukung fokus stok/promosi pada kategori unggulan dan evaluasi katalog kategori dengan performa rendah.

### Pertanyaan 3: Keterlambatan Pengiriman dan Kepuasan

In [ ]:
reviews_by_order = order_reviews_df.groupby("order_id", as_index=False)["review_score"].mean()
q3 = orders_period.merge(reviews_by_order, on="order_id", how="inner").merge(
    customers_df[["customer_id", "customer_state"]], on="customer_id", how="left"
)
q3 = q3.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date", "review_score"])
q3["delivery_delay"] = (q3["order_delivered_customer_date"] - q3["order_estimated_delivery_date"]).dt.days
q3["is_late"] = q3["delivery_delay"] > 0
q3["month"] = q3["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
monthly_delay = q3.groupby("month", as_index=False).agg(
    total_orders=("order_id", "nunique"), late_orders=("is_late", "sum")
)
monthly_delay["late_percentage"] = monthly_delay["late_orders"] / monthly_delay["total_orders"] * 100
state_delay = q3.groupby("customer_state", as_index=False).agg(
    total_orders=("order_id", "nunique"), late_orders=("is_late", "sum")
)
state_delay["late_percentage"] = state_delay["late_orders"] / state_delay["total_orders"] * 100
state_delay_qualified = state_delay[state_delay["total_orders"] >= 30].sort_values("late_percentage", ascending=False)
display(monthly_delay)
display(state_delay_qualified.head(10))

,month,total_orders,late_orders,late_percentage
0,2016-09-01,1,1,100.000000
1,2016-10-01,267,2,0.749064
2,2016-12-01,1,0,0.000000
3,2017-01-01,741,21,2.834008
4,2017-02-01,1643,42,2.556299
5,2017-03-01,2527,113,4.471706
6,2017-04-01,2290,147,6.419214
7,2017-05-01,3517,99,2.814899
8,2017-06-01,3111,92,2.957248
9,2017-07-01,3842,105,2.732952


,customer_state,total_orders,late_orders,late_percentage
1,AL,394,82,20.812183
9,MA,712,122,17.134831
24,SE,334,50,14.970060
16,PI,471,65,13.800425
5,CE,1273,174,13.668500
21,RR,41,5,12.195122
18,RJ,12214,1456,11.920747
4,BA,3229,384,11.892227
13,PA,933,100,10.718114
7,ES,1969,205,10.411376


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.lineplot(data=monthly_delay, x="month", y="late_percentage", marker="o", ax=axes[0], color="#dc2626")
axes[0].set_title("Persentase Pesanan Terlambat per Bulan")
axes[0].set_xlabel("Bulan"); axes[0].set_ylabel("Pesanan terlambat (%)")
axes[0].tick_params(axis="x", rotation=45)
state_plot = state_delay_qualified.head(10).sort_values("late_percentage")
sns.barplot(data=state_plot, x="late_percentage", y="customer_state", ax=axes[1], color="#f97316")
axes[1].set_title("State dengan Tingkat Keterlambatan Tertinggi")
axes[1].set_xlabel("Pesanan terlambat (%)"); axes[1].set_ylabel("State pelanggan")
fig.tight_layout(); plt.show()

/var/folders/69/x82gzmxd4f5_74mx5vv2423w0000gn/T/ipykernel_12207/3558851489.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


In [ ]:
delivery_summary = q3.groupby("is_late").agg(
    total_orders=("order_id", "nunique"), average_review=("review_score", "mean"), average_delay=("delivery_delay", "mean")
).round(2)
correlation = q3[["delivery_delay", "review_score"]].corr().iloc[0, 1]
print("Ringkasan pengiriman:")
display(delivery_summary)
print(f"Korelasi keterlambatan dengan review score: {correlation:.3f}")
print("State dengan tingkat keterlambatan tertinggi (minimal 30 pesanan):", state_delay_qualified.iloc[0]["customer_state"])

Ringkasan pengiriman:


,total_orders,average_review,average_delay
is_late,,,
False,89448,4.29,-13.51
True,6382,2.27,10.52


Korelasi keterlambatan dengan review score: -0.267
State dengan tingkat keterlambatan tertinggi (minimal 30 pesanan): AL


**Insight Pertanyaan 3:** Persentase keterlambatan berubah-ubah tiap bulan dan paling tinggi pada **Maret 2018 (18,65%)** dalam periode yang dianalisis. Setelah menyaring state dengan minimal 30 pesanan agar persentase dari sampel sangat kecil tidak menyesatkan, **AL** memiliki tingkat keterlambatan tertinggi (**20,81% dari 394 pesanan**). Pesanan terlambat memiliki rata-rata review **2,27**, dibandingkan **4,29** untuk pesanan tepat waktu; korelasi keterlambatan dan review score sebesar **-0,267**, sehingga temuan ini menunjukkan hubungan negatif, bukan bukti sebab-akibat.

## Analisis Lanjutan (Opsional)

Analisis lanjutan pada Pertanyaan 3 menghubungkan ketepatan pengiriman dengan review score dan membandingkan tingkat keterlambatan antarwilayah.


## Conclusion & Recommendation

1. **Pertanyaan 1:** Tren penjualan dan jumlah pesanan meningkat dari 2017 hingga awal 2018. Penurunan terbesar secara numerik terjadi pada September 2018 sebesar R$ 854.541,33, tetapi bulan tersebut tidak lengkap sehingga tidak layak dijadikan dasar keputusan performa. **Action item:** gunakan perbandingan bulan penuh dan tandai periode parsial pada dashboard sebelum menetapkan target penjualan.
2. **Pertanyaan 2:** `health_beauty` menghasilkan penjualan tertinggi (R$ 1.258.681,34), sedangkan `bed_bath_table` memiliki pesanan terbanyak (9.417). `security_and_services` terendah pada keduanya (R$ 283,29 dan 2 pesanan). **Action item:** prioritaskan stok dan promosi kategori unggulan, lalu evaluasi harga, visibilitas, atau kelayakan katalog kategori terbawah.
3. **Pertanyaan 3:** Keterlambatan tertinggi terjadi pada Maret 2018 (18,65%), dan state AL tertinggi di antara state dengan minimal 30 pesanan (20,81%). Review pesanan terlambat lebih rendah (2,27 vs 4,29), dengan korelasi -0,267. **Action item:** audit mitra/rute logistik di AL dan lakukan notifikasi proaktif untuk pesanan yang berisiko terlambat; evaluasi dampaknya dengan metrik SLA bulanan.